# AgensFlow LangGraph — quickstart

This notebook boots the whole free-tier bundle **in one Python kernel** and
runs a real MAS end-to-end:

1. Start the AgensFlow policy server in-process (no separate `uvicorn`)
2. Issue a user API key
3. Import a converged starter policy — the substrate skips cold-start
4. Build the `parallel_critic_mas` graph
5. Run a real query against OpenRouter
6. Inspect the routing decisions the substrate made

**Prerequisites:** `OPENROUTER_API_KEY` in your environment. That's it — no
hosted server, no auth setup, no external DB.

The whole notebook runs top-to-bottom in ~2 minutes and costs ~$0.30 in
OpenRouter fees for the one demo query.

## 1. Boot the policy server in-process

We use `httpx.ASGITransport` to run the FastAPI app inside this Python
process — same trick our integration tests use. All `/langgraph/*` requests
go through the ASGI transport instead of real HTTP, but everything else
(bandits, storage, tenant isolation) works exactly like a real deployment.

In [ ]:
import os

# Force SQLite-in-memory for this notebook run so we don't need ./data/
os.environ.setdefault('AGF_DATABASE_URL', 'sqlite+aiosqlite:///:memory:')
os.environ.setdefault('AGF_ENV', 'test')
os.environ.setdefault('AGF_JWT_SECRET', 'notebook-not-for-production')

from httpx import ASGITransport, AsyncClient
from asgi_lifespan import LifespanManager

from agensflow_mcp.app import create_app
from agensflow_mcp.db.session import init_db, get_engine
from agensflow_mcp.db.models import Base

app = create_app()
await init_db()
engine = get_engine()
async with engine.begin() as conn:
    await conn.run_sync(Base.metadata.create_all)

lifespan_mgr = LifespanManager(app)
await lifespan_mgr.__aenter__()
server_client = AsyncClient(transport=ASGITransport(app=app), base_url='http://test')
print('  ✓ policy server booted in-process')

## 2. Issue an anonymous API key

The same endpoint a real deployment exposes — `POST /auth/anonymous`.

In [ ]:
resp = await server_client.post('/auth/anonymous')
api_key = resp.json()['api_key']
print(f'  api_key: {api_key[:20]}...')

## 3. Route the adapter's HTTP calls through our in-process server

Normally `agensflow-langgraph`'s client hits a real HTTPS server. Here we
monkey-patch it to use the in-process ASGI transport — one small class
override, then all the decorator's `/langgraph/*` calls flow through the
notebook's own kernel.

In [ ]:
from agensflow_langgraph import client as agf_client

class _NotebookClient(agf_client.AgensFlowClient):
    async def _a_post_model(self, path, payload, model_cls):
        r = await server_client.post(path, json=payload, headers=self._headers)
        self._raise_for_status(r)
        return model_cls.model_validate(r.json())

agf_client._CACHE.clear()
agf_client.AgensFlowClient = _NotebookClient
os.environ['AGENSFLOW_SERVER_URL'] = 'http://test'
os.environ['AGENSFLOW_API_KEY'] = api_key
print('  ✓ adapter wired to in-process server')

## 4. Import the converged starter policy

`parallel_critic_v1.json` is the exported bandit state from 40 real runs of
our example MAS. Importing it gives the substrate warm priors on the arms —
no cold-start exploration needed for signatures whose names match ours.

In [ ]:
from pathlib import Path
from agensflow_langgraph import aimport_policy

# Path is relative to the repo root — the notebook lives in notebooks/
policy_path = Path('..') / 'examples' / 'starter_policies' / 'parallel_critic_v1.json'
assert policy_path.exists(), f'Starter policy not found at {policy_path.resolve()}'

result = await aimport_policy(policy_path)
print(f'  ✓ imported {result["signatures_merged"]} signatures / '
      f'{result["actions_merged"]} arms into the substrate')

## 5. Build the parallel_critic MAS graph

This is the same graph the starter policy was trained on — six
@agensflow-decorated nodes, critic + verifier running in parallel.

In [ ]:
import sys
sys.path.insert(0, str(Path('..').resolve()))

if not os.environ.get('OPENROUTER_API_KEY'):
    raise RuntimeError(
        'Set OPENROUTER_API_KEY before running the query cell below.\n'
        'This example makes real LLM calls; ~$0.30 per query.'
    )

from examples.parallel_critic_mas.graph import build_pools, build_graph

pools = build_pools()
compiled = build_graph(pools)
print('  ✓ graph compiled')
print(f'  nodes: {list(compiled.get_graph().nodes)}')

## 6. Run one real query end-to-end

The substrate picks per-node actions based on the imported priors + UCB1
exploration bonus. All 6 nodes route through the in-process server.

In [ ]:
user_task = (
    'What is the difference between TCP and UDP, and when should each be used? '
    'Answer using only the provided documents.'
)

result = await compiled.ainvoke(
    {'user_task': user_task, 'trace': []},
    config={'configurable': {'thread_id': 'notebook_demo_1'}},
)

print('  Final answer:')
print(f'    {result["final_answer"]}')
print()
print('  Substrate routed the graph as:')
for step in result['trace']:
    print(f'    {step["node"]:<10} → {step["action"]}')

## 7. Inspect the routing decisions server-side

Every decision the substrate made — with real token counts, latency, and
the reward we submit (below) — landed in the server's `decision_records`
table. Query `/langgraph/decisions` to see them.

In [ ]:
from agensflow_langgraph import arecord_reward

# Submit a placeholder reward so the decisions show up as 'rewarded'
await arecord_reward(quality=1.0, thread_id='notebook_demo_1')

resp = await server_client.get(
    '/langgraph/decisions?limit=10',
    headers={'Authorization': f'Bearer {api_key}'},
)
for d in resp.json()['decisions']:
    print(f'  {d["signature"]:<10} action={d["action"]:<10} '
          f'tokens=in{d["tokens_input"] or 0}/out{d["tokens_output"] or 0}  '
          f'lat={d["latency_s"]:.1f}s  status={d["status"]}')

## 8. Cleanup

Close the in-process server.

In [ ]:
await server_client.aclose()
await lifespan_mgr.__aexit__(None, None, None)
print('  ✓ done')

## What you just saw

In this one notebook you:

- Booted a full AgensFlow policy server (bandits, tenant isolation, storage)
  inside the Python kernel
- Issued a real API key
- Warm-started the substrate from a converged 40-run policy
- Built a real MAS with parallel critic + verifier
- Ran one query end-to-end through OpenRouter with real token counting
- Verified the substrate recorded every routing decision with cost/latency

**That's the free-tier bundle.** For production, swap in a real `uvicorn`
server + Postgres, keep the exact same client + graph code — the ASGI
transport in cell 1 is the only thing that changes.

**Next:** see [`docs/integration.md`](../docs/integration.md) for the
advanced patterns (streaming, checkpointer, redaction, custom judges) and
[`examples/`](../examples/) for two more topology variations.